<a href="https://colab.research.google.com/github/DDDBBBPPP/DDDBBBPPP.github.io/blob/main/Streaming_WAF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica: Streaming + WAF

In [6]:
import time
import random
from datetime import datetime
from collections import deque

## Productor

In [7]:
class Streaming:
  def __init__(self):
    self.usuarios = ["David", "Antonio", "Alejandra", "Julio", "María", "Jaime", "Ezequiel", "Rosario"]
    self.endpoints = ["login", "logout", "admin", "me", "home"]

  def generar_evento(self):

        return {
            'timestamp': datetime.now().isoformat(),
            'event_time': time.time(),
            'user_id': random.choice(self.usuarios),
            'endpoint': random.choice(self.endpoints),
        }

  def stream(self, n_eventos=100, delay_max=1.0):
    """Genera n eventos con delay aleatorio"""
    for i in range(n_eventos):
      evento = self.generar_evento()
      yield evento
      time.sleep(random.uniform(0.1, delay_max))

## WAF: detección y baneo

Reglas: más de 3 `login` o más de 2 `admin` por usuario en 10 segundos. Baneo de 10 segundos.

In [8]:
from collections import deque

class FraudeDetector:

    def __init__(self, max_logins=3, max_admin=2, tiempo_baneo=10):
        self.max_logins = max_logins
        self.max_admin = max_admin
        self.tiempo_baneo = tiempo_baneo

        self.user_history = {}
        self.baneados = {}

    def check(self, evento):
        alertas = []
        usuario = evento['user_id']
        endpoint = evento['endpoint']
        current_time = evento['event_time']

        # Comprobar si está baneado
        if usuario in self.baneados:
            restante = self.baneados[usuario] - current_time

            if restante > 0:
                alertas.append(f"🚫 {usuario} bloqueado. Quedan {restante:.1f} segundos")
                return False, alertas

            del self.baneados[usuario]
            self.user_history.pop(usuario, None)
            alertas.append(f"✅ {usuario} ya no está baneado")

        # Guardar historial por usuario y endpoint
        if usuario not in self.user_history:
            self.user_history[usuario] = {}

        if endpoint not in self.user_history[usuario]:
            self.user_history[usuario][endpoint] = deque()

        historial = self.user_history[usuario][endpoint]

        # Eliminar eventos de hace más de 10 segundos
        while historial and current_time - historial[0] > 10:
            historial.popleft()

        historial.append(current_time)

        # Regla 1: Demasiados logins
        if endpoint == "login" and len(historial) > self.max_logins:
            alertas.append(f"⚠️ DEMASIADOS LOGINS: {usuario} - {len(historial)} intentos")

        # Regla 2: Demasiados accesos a admin
        if endpoint == "admin" and len(historial) > self.max_admin:
            alertas.append(f"⚠️ DEMASIADOS ACCESOS A ADMIN: {usuario} - {len(historial)} intentos")

        # Si hay alertas, banear
        if alertas:
            self.baneados[usuario] = current_time + self.tiempo_baneo
            alertas.append(f"🚫 {usuario} baneado durante {self.tiempo_baneo} segundos")
            return False, alertas

        return True, alertas

## Consumidor de estadísticas

Recibe únicamente eventos permitidos y muestra las estadísticas una sola vez al final.

In [9]:
class Consumidor:
    def __init__(self):
        self.eventos = []

    def recibir(self, evento):
        self.eventos.append(evento)

    def mostrar_estadisticas(self):
        total = len(self.eventos)
        endpoints = {}
        usuarios = {}

        for evento in self.eventos:
            endpoint = evento['endpoint']
            usuario = evento['user_id']
            endpoints[endpoint] = endpoints.get(endpoint, 0) + 1
            usuarios[usuario] = usuarios.get(usuario, 0) + 1

        porcentajes = {
            endpoint: round(cantidad / total * 100, 2)
            for endpoint, cantidad in endpoints.items()
        } if total else {}

        usuarios_ordenados = dict(sorted(usuarios.items(), key=lambda x: x[1], reverse=True))

        print("\n=== ESTADÍSTICAS FINALES ===")
        print(f"Total de eventos permitidos: {total}")
        print(f"Porcentaje de endpoints: {porcentajes}")
        print(f"Usuarios más activos: {usuarios_ordenados}")

## 4. Demostración completa

**Productor → WAF → Consumidor** (permitidos) o **registro de bloqueos**. Se ven las peticiones y alertas durante la simulación.
Estadísticas al final.

In [10]:
print("SIMULACIÓN STREAMING\n")

simulator = Streaming()
waf = FraudeDetector()
consumidor = Consumidor()
eventos_bloqueados = []

for evento in simulator.stream(n_eventos=100, delay_max=0.5):
    permitido, alertas = waf.check(evento)

    if permitido:
        consumidor.recibir(evento)
        print(f"✅ {evento['user_id']} → {evento['endpoint']}")
    else:
        eventos_bloqueados.append(evento)
        print(f"🚫 {evento['user_id']} → {evento['endpoint']} (BLOQUEADO)")

    for alerta in alertas:
        print(alerta)

consumidor.mostrar_estadisticas()
print(f"Eventos bloqueados guardados: {len(eventos_bloqueados)}")

SIMULACIÓN STREAMING

✅ Jaime → admin
✅ David → admin
✅ Jaime → login
✅ Ezequiel → home
✅ Rosario → admin
✅ Alejandra → admin
✅ Julio → home
✅ David → me
✅ Julio → me
✅ David → logout
✅ David → logout
✅ Julio → logout
✅ Alejandra → logout
✅ Antonio → logout
✅ María → admin
✅ Julio → me
✅ David → login
✅ María → home
✅ David → me
✅ Ezequiel → logout
✅ Jaime → login
✅ María → me
✅ Ezequiel → logout
✅ Jaime → home
✅ Rosario → me
✅ Julio → home
✅ Julio → me
✅ Antonio → login
✅ Antonio → admin
✅ Julio → me
✅ Julio → home
✅ Jaime → me
✅ Ezequiel → home
✅ María → home
✅ David → me
✅ María → logout
✅ Ezequiel → login
✅ Julio → admin
✅ Antonio → logout
✅ Alejandra → me
✅ Alejandra → me
✅ Julio → login
✅ María → me
✅ Rosario → admin
✅ Alejandra → logout
✅ Alejandra → me
✅ Alejandra → admin
✅ María → login
✅ María → admin
✅ Rosario → admin
✅ Ezequiel → me
✅ Ezequiel → me
✅ Alejandra → admin
✅ Alejandra → logout
✅ Rosario → login
✅ Alejandra → logout
✅ Julio → admin
✅ Ezequiel → logout
✅ Jaime → a